# Entrenamiento independiente de B4

Este cuaderno ejecuta solo B4 para los escenarios A, B y C con todas las partidas de `train`. Guarda estudios, métricas y modelos en `MyDrive/Applied-AI-Agent-Project/model_checkpoints/b4_independiente/`. Sus checkpoints son independientes de B3 y del pipeline secuencial. Si Colab se interrumpe, vuelve a ejecutar setup y entrenamiento para reanudar. `val` compara finalistas; `test` no se toca.

Para empezar, la persona que lo ejecute debe iniciar sesión en Google, permitir el montaje de su propio Drive y, si el dataset no existe en `/content/datos`, configurar sus credenciales/permisos de Kaggle para descargarlo. Abrir el enlace por sí solo no inicia el entrenamiento.

In [ ]:
from pathlib import Path
import sys
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints/b4_independiente')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo pull origin 04MainAIModelTrainingAndDevelopment
%cd /content/repo
!pip install -q -r requirements.txt
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
from sklearn import __version__ as sklearn_version
assert tuple(map(int, sklearn_version.split('.')[:2])) >= (1, 7), 'B4 necesita scikit-learn >=1.7; reinicia el runtime.'
DATA_DIR = Path('/content/datos')
from ptcg_wp.ingest import load_days
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(ruta.exists() for ruta in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(ruta.exists() for ruta in esperados), 'Faltan tablas: revisa Kaggle y vuelve a ejecutar setup.'
print('Entorno B4 listo | sklearn:', sklearn_version, '| checkpoints:', CHECKPOINT_ROOT)

## Presupuesto de búsqueda

La configuración inicial usa 5 trials por métrica, cinco pliegues agrupados y un máximo de 500 iteraciones con parada temprana. El límite de una hora por métrica se comprueba entre trials, así que un ajuste ya iniciado puede terminar después de ese límite. El materializado de datos B4 se reconstruye en el disco local de cada runtime.

In [ ]:
from tempfile import TemporaryDirectory
import gc
import faulthandler
import pandas as pd
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.b4 import materialize_split, b4_optuna_cv, fit_b4_candidates_val
from ptcg_wp.checkpoints import atomic_csv
from ptcg_wp.features import feature_columns
import pyarrow.parquet as pq

features = feature_columns(pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names)
B4_TRIALS = 5
B4_MAX_ITER = 500
B4_PATIENCE = 30
B4_TIMEOUT_SECONDS = 3600
MAX_RAM_GIB = 10.0
SEED = 2026

trace_file = open('/content/b4_pipeline_tracebacks.log', 'a', buffering=1)
faulthandler.enable(file=trace_file)
faulthandler.dump_traceback_later(600, repeat=True, file=trace_file)
b4_outputs = {}
try:
    for scenario in ('A', 'B', 'C'):
        output_dir = CHECKPOINT_ROOT / 'efficient_v2' / f'escenario_{scenario}'
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f'B4 {scenario}: preparando train y val', flush=True)
        with TemporaryDirectory(prefix=f'b4-{scenario}-', dir='/content') as cache:
            stream = ScenarioStream(DATA_DIR, scenario, tuple(features), batch_size=32768)
            train = materialize_split(stream, features, 'train', cache)
            val = materialize_split(stream, features, 'val', cache)
            try:
                cv, candidates = b4_optuna_cv(
                    train, n_splits=5, n_trials=B4_TRIALS, seed=SEED,
                    max_iter=B4_MAX_ITER, patience=B4_PATIENCE,
                    max_ram_gib=MAX_RAM_GIB, timeout=B4_TIMEOUT_SECONDS,
                    output_dir=output_dir,
                    study_cache_dir=f'/content/ptcg_optuna/b4_{scenario}',
                    verbose=True,
                )
                atomic_csv(candidates.to_dict('records'), output_dir / 'b4_candidatas.csv')
                val_metrics = fit_b4_candidates_val(
                    train, val, candidates, seed=SEED, max_iter=B4_MAX_ITER,
                    patience=B4_PATIENCE, max_ram_gib=MAX_RAM_GIB,
                    output_dir=output_dir, scenario=scenario, verbose=True,
                )
                atomic_csv(val_metrics.to_dict('records'), output_dir / 'b4_val_finalistas.csv')
                b4_outputs[scenario] = str(output_dir)
                print(f'B4 {scenario}: terminado | candidatas={len(candidates)} | {output_dir}', flush=True)
                del cv, candidates, val_metrics, stream
            finally:
                train.close()
                val.close()
        gc.collect()
finally:
    faulthandler.cancel_dump_traceback_later()
    faulthandler.disable()
    trace_file.close()
print('B4 listo. Checkpoints:', b4_outputs)